# Example 1: solve Eq. 19 for the alpha coefficients

This notebook solves the Eq. 19 linear system from `SG_paper.tex` using the same approach used for Eq. 15 in `SG_solver.ipynb`: build the RBF matrix `A`, build the divided-difference RHS, then solve `A alpha = rhs` with `solve_spd`.

In [8]:
import numpy as np
import matplotlib.pyplot as plt

from SPD_solver import solve_spd
from SG_solver import rbf_matrix, second_divided_difference, d2_phi, d2_L_W2

In [ ]:
# Paper Example 1 parameters.
a = -1.0
b = 1.0
h = 0.01
K = 8
s = 0.8
c = 0.027 
tau = h

n_float = (b - a) / h
n = int(round(n_float))
if not np.isclose(n_float, n):
    raise ValueError("h must divide b-a exactly.")

if n % K != 0:
    raise ValueError("n must be divisible by K because the paper defines N = n / K.")

N = n // K
x = np.linspace(a, b, n + 1)

print(f"n = {n}, N = {N}")

n = 200, N = 25


In [11]:
# Choose the interpolation centers exactly as grid indices, so x_{k_i-1}, x_{k_i}, x_{k_i+1} exist.
# This follows the same center-index approach used in SG_solver.ipynb.
k_idx = np.rint(np.linspace(1, n - 1, N)).astype(int)
k_idx[0] = 1
k_idx[-1] = n - 1

if len(np.unique(k_idx)) != N:
    raise ValueError("Center selection produced duplicate indices; increase n or reduce K.")

xk = x[k_idx]

print("First five center indices:", k_idx[:5])
print("Last five center indices:", k_idx[-5:])
print("First five centers:", xk[:5])
print("Last five centers:", xk[-5:])

First five center indices: [ 1  9 18 26 34]
Last five center indices: [166 174 182 191 199]
First five centers: [-0.99 -0.91 -0.82 -0.74 -0.66]
Last five centers: [0.66 0.74 0.82 0.91 0.99]


In [12]:
# Analytical solution from paper Example 1, used here only to supply one time-level vector u^n.
def exact_u(x, t):
    return 0.5 * (np.sin(np.pi * (x + t)) + np.sin(np.pi * (x - t)))


t_n = 0.0
u_n = exact_u(x, t_n)

# For t_n = 0, this is sin(pi*x), so u_xx = -pi^2 sin(pi*x).
u_xx_exact_at_centers = -(np.pi**2) * np.sin(np.pi * xk)

$$
\sum_{j=1}^{N} \alpha_j \varphi(|x_{k_i}-x_{k_j}|) =
\frac{2u_{k_i+1}^{n}}{(x_{k_i+1}-x_{k_i})(x_{k_i+1}-x_{k_i-1})}
+ \frac{2u_{k_i-1}^{n}}{(x_{k_i}-x_{k_i-1})(x_{k_i+1}-x_{k_i-1})}
- \frac{2u_{k_i}^{n}}{(x_{k_i}-x_{k_i-1})(x_{k_i+1}-x_{k_i})}.
\tag{19}$$

The right side is the same three-point second divided-difference formula from Eq. 15, and applied at the current time-level values $u_n$.
$$ \alpha_m = \sum_{i=1}^{N}\left[A^{-1}\right]_{mi}\left(\frac{2u_{k_i+1}^{n}}{(x_{k_i+1}-x_{k_i})(x_{k_i+1}-x_{k_i-1})} + \frac{2u_{k_i-1}^{n}}{(x_{k_i}-x_{k_i-1})(x_{k_i+1}-x_{k_i-1})} - \frac{2u_{k_i}^{n}}{(x_{k_i}-x_{k_i-1})(x_{k_i+1}-x_{k_i})} \right),\\ \quad m=1,\ldots,N,$$
$$A_{ij}=\varphi(|x_{k_i}-x_{k_j}|)$$

In [13]:
A = rbf_matrix(xk, s)
rhs = []
for kj in k_idx:
    rhs.append(second_divided_difference(x, u_n, kj))
rhs = np.asarray(rhs)

alpha = solve_spd(A, rhs)

print(alpha[:5])

[  -760.31056011   3274.24269362  -8983.70831887  17039.17440759
 -24061.80192955]


$$u_{k}^{n+1} = 2u_{k}^{n} - u_{k}^{n-1} - \tau^{2} \sin(u_{k}^{n}) + \tau^{2} \left[ \sum_{j=1}^{N} \alpha_{j} \varphi(x_{k} - x_{k_{j}}) + \sum_{i=0}^{n} \left( u_{i}^{n} - \sum_{j=1}^{N} \alpha_{j} \sqrt{s^{2} + (x_{i} - x_{k_{j}})^{2}} \right) \psi_{i}''(x_{k}) \right], \tag{18}$$